In [1]:
spark.sparkContext.setLogLevel("ERROR")

In [2]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|   bronze|
|  default|
|     gold|
|   silver|
+---------+



In [3]:
%load_ext sparksql_magic

In [4]:
%%sparksql
show tables

namespace,tableName,isTemporary
default,table_a,False
default,table_b,False


In [5]:
%%sparksql
CREATE TABLE IF NOT EXISTS table_a (
    id BIGINT,
    name STRING,
    age INT,
    department STRING
)
USING DELTA
LOCATION 's3a://lakehouse/default/table_a'

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [6]:
%%sparksql
INSERT INTO table_a VALUES
    (1, 'Alice', 25, 'Engineering'),
    (2, 'Bob', 30, 'Data Engineering'),
    (3, 'Charlie', 28, 'Analytics');

In [ ]:
%%sparksql
CREATE TABLE IF NOT EXISTS table_b (
    id BIGINT,
    name STRING,
    age INT,
    department STRING
)
USING DELTA
LOCATION 's3a://lakehouse/default/table_b'

In [15]:
%%sparksql
INSERT INTO table_b
SELECT *
FROM table_a;

26/10/23 16:36:25 ERROR ContextFactory: Query execution is null: can't emit event for executionId 51


In [7]:
%%sparksql
CREATE TABLE IF NOT EXISTS table_c (
    id BIGINT,
    name STRING,
    age INT,
    department STRING
)
USING DELTA
LOCATION 's3a://lakehouse/default/table_c'

In [9]:
%%sparksql
INSERT INTO table_c
SELECT *
FROM table_a
union all
SELECT *
FROM table_b

In [25]:
# %%sparksql
# --DROP DATABASE IF EXISTS telco CASCADE

In [29]:
%%sparksql
show tables in gold

namespace,tableName,isTemporary
gold,sales,False


In [30]:
%%sparksql
desc history gold.sales

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-10-19 14:11:23,airflow,airflow,CREATE OR REPLACE TABLE AS SELECT,{'queryId': '20261019_141122_00046_gzp6c'},null,null,trino-483-trino,0,WriteSerializable,True,{},null,null
2,2026-10-19 14:10:49,airflow,airflow,CREATE OR REPLACE TABLE AS SELECT,{'queryId': '20261019_141048_00023_gzp6c'},null,null,trino-483-trino,0,WriteSerializable,True,{},null,null
1,2026-10-19 14:00:11,airflow,airflow,CREATE OR REPLACE TABLE AS SELECT,{'queryId': '20261019_140011_00044_vacgn'},null,null,trino-483-trino,0,WriteSerializable,True,{},null,null
0,2026-10-19 13:59:59,airflow,airflow,CREATE OR REPLACE TABLE AS SELECT,{'queryId': '20261019_135959_00022_vacgn'},null,null,trino-483-trino,0,WriteSerializable,True,{},null,null


In [27]:
%%sparksql
select * from gold.sales

[Stage 10:>                                                         (0 + 1) / 1]

only showing top 20 row(s)


row_id,order_id,order_date,ship_date,days_to_ship,ship_mode,customer_id,customer_name,segment,country,region,state,city,product_id,category,sub_category,product_name,sales,quantity,discount,profit
13,CA-2017-114412,2017-04-15,2017-04-20,5,Standard Class,AA-10480,Andrew Allen,Consumer,United States,South,North Carolina,Concord,OFF-PA-10002365,Office Supplies,Paper,Xerox 1967,15.552,3,0.2,5.4432
38,CA-2015-117415,2015-12-27,2015-12-31,4,Standard Class,SN-20710,Steve Nguyen,Home Office,United States,Central,Texas,Houston,OFF-EN-10002986,Office Supplies,Envelopes,"#10-4 1/8"" x 9 1/2"" Premium Diagonal Seam Envelopes",113.328,9,0.2,35.415
44,CA-2017-139619,2017-09-19,2017-09-23,4,Standard Class,ES-14080,Erin Smith,Corporate,United States,South,Florida,Melbourne,OFF-ST-10003282,Office Supplies,Storage,"Advantus 10-Drawer Portable Organizer, Chrome Metal Frame, Smoke Drawers",95.616,2,0.2,9.5616
54,CA-2016-105816,2016-12-11,2016-12-17,6,Standard Class,JM-15265,Janet Molinari,Corporate,United States,East,New York,New York City,OFF-FA-10000304,Office Supplies,Fasteners,Advantus Push Pins,15.26,7,0.0,6.2566
60,CA-2016-111682,2016-06-17,2016-06-18,1,First Class,TB-21055,Ted Butterfield,Consumer,United States,East,New York,Troy,TEC-AC-10002167,Technology,Accessories,Imation 8gb Micro Traveldrive Usb 2.0 Flash Drive,30.0,2,0.0,3.3
61,CA-2016-111682,2016-06-17,2016-06-18,1,First Class,TB-21055,Ted Butterfield,Consumer,United States,East,New York,Troy,OFF-BI-10001460,Office Supplies,Binders,Plastic Binding Combs,48.48,4,0.2,16.362
70,CA-2016-119823,2016-06-04,2016-06-06,2,First Class,KD-16270,Karen Daniels,Consumer,United States,South,Virginia,Springfield,OFF-PA-10000482,Office Supplies,Paper,"Snap-A-Way Black Print Carbonless Ruled Speed Letter, Triplicate",75.88,2,0.0,35.6636
76,US-2017-118038,2017-12-09,2017-12-11,2,First Class,KB-16600,Ken Brennan,Corporate,United States,Central,Texas,Houston,OFF-BI-10004182,Office Supplies,Binders,Economy Binders,1.248,3,0.8,-1.9344
83,CA-2014-139451,2014-10-12,2014-10-16,4,Standard Class,DN-13690,Duane Noonan,Consumer,United States,West,California,San Francisco,OFF-ST-10002370,Office Supplies,Storage,"Sortfiler Multipurpose Personal File Organizer, Black",21.39,1,0.0,6.2031
106,US-2015-156867,2015-11-13,2015-11-17,4,Standard Class,LC-16870,Lena Cacioppo,Consumer,United States,West,Colorado,Aurora,OFF-BI-10002794,Office Supplies,Binders,"Avery Trapezoid Ring Binder, 3"" Capacity, Black, 1040 sheets",36.882,3,0.7,-25.8174
